In [1]:
import pandas as pd

In [2]:
%run ../gs_slimming.py
print("="*120)
print("gs_slimming done.")
print("="*120)
%run flattening.py
print("="*120)
print("flattening done.")
print("="*120)

Mismatches: {'ViacomCBS_ESG Report_2020-2021_vFINAL.pdf'}

No. of gold_standard reports: 139

No. of downloadable reports: 113

No. of reports in gs_slim: 54

status
partial     45
complete     9
Name: count, dtype: int64

scopes_present
1+2lb          18
1+2lb+3        15
1+2lb+2mb+3     9
1+2lb+2mb       6
1+2mb+3         1
1+3             1
1               1
2lb+3           1
2mb             1
3               1
Name: count, dtype: int64
gs_slimming done.

  Flattening 54 JSONs => /Users/jannikkuhlmann/VSC/LaTeX/2026_BA_Code/evaluations/PipelineB/PipelineB-Answers/1st_Qwen3-VL-30B-A3B-Thinking.csv
  Reports    : 54
  Zeilen     : 672
  Fehler     : 0
  Gespeichert: /Users/jannikkuhlmann/VSC/LaTeX/2026_BA_Code/evaluations/PipelineB/PipelineB-Answers/1st_Qwen3-VL-30B-A3B-Thinking.csv

  Flattening 54 JSONs => /Users/jannikkuhlmann/VSC/LaTeX/2026_BA_Code/evaluations/PipelineB/PipelineB-Answers/1st_Qwen3-VL-32B-Instruct.csv
  Reports    : 54
  Zeilen     : 706
  Fehler     : 0
  Gespeich

## Import slimmed down Gold-Standard

In [3]:
gs = pd.read_csv("../gs_slim.csv")
gs["year"] = gs["year"].astype(str) #Correction needed for e.g. fiscal years "FY 2021/2022"

## Import all 3x2=6 Extractions

In [4]:
think1 = pd.read_csv("./PipelineB-Answers/1st_Qwen3-VL-32B-Thinking.csv")
think2 = pd.read_csv("./PipelineB-Answers/2nd_Qwen3-VL-32B-Thinking.csv")

moe1   = pd.read_csv("./PipelineB-Answers/1st_Qwen3-VL-30B-A3B-Thinking.csv")
moe2   = pd.read_csv("./PipelineB-Answers/2nd_Qwen3-VL-30B-A3B-Thinking.csv")

instr1 = pd.read_csv("./PipelineB-Answers/1st_Qwen3-VL-32B-Instruct.csv")
instr2 = pd.read_csv("./PipelineB-Answers/2nd_Qwen3-VL-32B-Instruct.csv")

df_to_merge = [
    (think1, "_t1"),
    (think2, "_t2"),
    (moe1, "_m1"),
    (moe2, "_m2"),
    (instr1, "_i1"),
    (instr2, "_i2")
]

## Matching Extractions to Gold-Standard Scheme

In [6]:
scope_mapping = {
    "scope_1": "1",
    "scope_2_location_based": "2lb",
    "scope_2_market_based" : "2mb",
    "scope_3" : "3"
}    

# Prints out only if sth. goes wrong
for df, sf in df_to_merge:
    # Replace scope definitons to match Gold-Standard
    df['scope'] = df['scope'].replace(scope_mapping)
    
    # Ensure every value column is a float64 to match Gold-Standard
    converted = pd.to_numeric(
        df['value'].astype(str).str.replace(",", "", regex=False),
        errors="coerce"
    )
    failed = df['value'][converted.isna() & df['value'].notna()]
    if len(failed) > 0:
        print(f"{sf}: {len(failed)} nicht konvertierbar: {failed.unique()}")
    df['value'] = converted


Checking if all `report_names` are the same

And checking if all `report_names` from the extractions are exactly matched in the GoldStandard

In [10]:
print("Think ok:", all(think1["report_name"].unique() == think2["report_name"].unique()))
print("Moe   ok:", all(think1["report_name"].unique() == think2["report_name"].unique()))
print("Instr ok:", all(think1["report_name"].unique() == think2["report_name"].unique()))
print()

# print(think1["report_name"].isin(gs["report_name"]).all())
# print(gs["report_name"].isin(think1["report_name"]).all())

in_think1_not_gs  = set(think1["report_name"]) - set(gs["report_name"])
in_gs_not_think1  = set(gs["report_name"])     - set(think1["report_name"])

print(f"In think1, nicht in GS: {len(in_think1_not_gs)} ==> {"OK" if len(in_think1_not_gs) == 0 else "NOK"}")
for r in sorted(in_think1_not_gs): print(" ", r)

print(f"\nIn GS, nicht in think1: {len(in_gs_not_think1)} ==> {"OK" if len(in_gs_not_think1) == 0 else "NOK"}")
for r in sorted(in_gs_not_think1): print(" ", r)

Think ok: True
Moe   ok: True
Instr ok: True

In think1, nicht in GS: 0 ==> OK

In GS, nicht in think1: 0 ==> OK


## Merging Extraction Values and Gold-Standard

In [11]:
merge_on = ["report_name", "scope", "year"]
agg_cols = ["value", "unit", "label"]

In [12]:
for df, sf in df_to_merge:
    dups = df.duplicated(subset=merge_on).sum()
    print(f"{sf}: {dups} doppelte Keys")


_t1: 147 doppelte Keys
_t2: 160 doppelte Keys
_m1: 202 doppelte Keys
_m2: 148 doppelte Keys
_i1: 190 doppelte Keys
_i2: 300 doppelte Keys


=> That's way I do not need a normal merge, but consolidate every extracted value, unit, and label from every dataframe (think1, think2, moe1, ...) value into a list.

Then I can validate if the gs-value is extracted or not.

These are represented as "objects" when calling merged.info() but are just python lists.

In [13]:
merged = gs.copy() # Setting a starting point for the loop. Everything has to be mapped to the Gold-Standard

for df, sf in df_to_merge:
    agg = (
        df.groupby(merge_on)[agg_cols]
        .agg(list)
        .reset_index()
        .rename(columns={col: f"{col}{sf}" for col in agg_cols})
    )
    merged = pd.merge(merged, agg, on=merge_on, how="left")
    
print(f"Zeilen: {len(merged)} (GS hatte {len(gs)})")
print(f"Spalten: {merged.columns.tolist()}")

Zeilen: 2208 (GS hatte 2208)
Spalten: ['report_name', 'year', 'scope', 'page', 'value', 'unit', 'unit_normalized', 'label', 'status', 'scopes_present', 'years_present', 'value_t1', 'unit_t1', 'label_t1', 'value_t2', 'unit_t2', 'label_t2', 'value_m1', 'unit_m1', 'label_m1', 'value_m2', 'unit_m2', 'label_m2', 'value_i1', 'unit_i1', 'label_i1', 'value_i2', 'unit_i2', 'label_i2']


## Saving created DataFrame

In [14]:
merged.to_csv("gs_extractions_raw.csv", index=False)
merged.to_json("gs_extractions_raw.json", index=False, orient="records", indent=4)

## To later read back the files:
# pd.read_csv("gs_extractions_raw.csv")   # Lists as Strings. Needs ast.literal_eval
# pd.read_json("gs_extractions_raw.json", orient="records")  # Lists instant usable


## Creating Overview which scopes are present for which years of a given report

In [15]:
year_scope_df = (
    gs[gs["value"].notna()]
    .groupby(["report_name", "year"])["scope"]
    .apply(lambda x: "+".join(sorted(x.unique())))
    .reset_index()
    .rename(columns={"scope": "scopes"})
)

year_scope_df["_sort"] = year_scope_df["report_name"].str.lower() # need to circumvent ASCII case-sensitive sorting
year_scope_df = year_scope_df.sort_values(["_sort", "year"], ignore_index=True).drop(columns="_sort")

year_scope_df.to_csv("scopes_per_year_and_report.csv", index=False)
year_scope_df.head()


,report_name,year,scopes
0,acuity brands inc_2022_report,2022,1+2lb+2mb+3
1,addtech_2022_report,2019,1+2lb+3
2,addtech_2022_report,2020,1+2lb+2mb+3
3,addtech_2022_report,2021,1+2lb+2mb+3
4,aixtron_2020_report,2019,1+2lb+3


In [16]:
year_scope_df[year_scope_df["report_name"] == "acuity brands inc_2022_report"]["scopes"]

0    1+2lb+2mb+3
Name: scopes, dtype: str